In [1]:
import numpy             as np
import matplotlib.pyplot as plt
import seaborn           as sns
import multiprocess      as mp
import pandas            as pd

from libraries                         import utils
from pymatgen.analysis.phase_diagram   import PhaseDiagram
from pymatgen.entries.computed_entries import ComputedEntry
from os                                import path
from sklearn                           import model_selection
from sklearn.ensemble                  import RandomForestRegressor
from sklearn.neural_network            import MLPRegressor
from sklearn.model_selection           import ShuffleSplit

#sns.set_theme()

Defining general output properties

In [2]:
linewidth      = 0.5
footnotesize   = 8
s_footnotesize = 10*0.8
eps_dpi        = 100
png_dpi        = 400
output_folder  = 'output/ternary_chalcohalides'
input_folder   = 'input/ternary_chalcohalides'

# Iterations might be run from here

In [3]:
miller_index = '010'

In [4]:
vbm     = np.loadtxt(f'{input_folder}/results_vbm_{miller_index}.txt')
bandgap = np.loadtxt(f'{input_folder}/results_band_gap.txt')

In [5]:
# Convert the array to a Pandas DataFrame
vbmd = pd.DataFrame(vbm, columns=["x", "y", "z", "VBM"])
vbmd.set_index(["x", "y", "z"], inplace=True)

bandgapd = pd.DataFrame(bandgap, columns=["x", "y", "z", "bandgap"])
bandgapd.set_index(["x", "y", "z"], inplace=True)

# Combine the DataFrames, aligning by index
data_010 = pd.concat([vbmd, bandgapd], axis=1)

data_010['CBM'] = data_010['VBM'] + data_010['bandgap']
data_010

VBM   bandgap       CBM
x    y    z                                 
0.00 0.00 0.00 -5.873304  1.524687 -4.348617
          0.05 -5.878601  1.529355 -4.349245
          0.10 -5.886466  1.532872 -4.353594
          0.15 -5.893398  1.536684 -4.356714
          0.20 -5.899609  1.542081 -4.357527
...                  ...       ...       ...
0.95 0.90 0.10       NaN  1.777273       NaN
     0.95 0.05       NaN  1.812115       NaN
1.00 0.20 0.15       NaN  1.357448       NaN
     0.25 0.20       NaN  1.373054       NaN
     0.90 0.05       NaN  1.755392       NaN

[1901 rows x 3 columns]

In [6]:
miller_index = '011'

In [7]:
vbm     = np.loadtxt(f'{input_folder}/results_vbm_{miller_index}.txt')
bandgap = np.loadtxt(f'{input_folder}/results_band_gap.txt')

In [8]:
# Convert the array to a Pandas DataFrame
vbmd = pd.DataFrame(vbm, columns=["x", "y", "z", "VBM"])
vbmd.set_index(["x", "y", "z"], inplace=True)

bandgapd = pd.DataFrame(bandgap, columns=["x", "y", "z", "bandgap"])
bandgapd.set_index(["x", "y", "z"], inplace=True)

# Combine the DataFrames, aligning by index
data_011 = pd.concat([vbmd, bandgapd], axis=1)

data_011['CBM'] = data_011['VBM'] + data_011['bandgap']
data_011

VBM   bandgap       CBM
x    y    z                                 
0.00 0.00 0.00 -6.283043  1.524687 -4.758356
          0.05 -6.297463  1.529355 -4.768107
          0.10 -6.309342  1.532872 -4.776469
          0.15 -6.316886  1.536684 -4.780202
          0.20 -6.324363  1.542081 -4.782282
...                  ...       ...       ...
0.95 0.90 0.10       NaN  1.777273       NaN
     0.95 0.05       NaN  1.812115       NaN
1.00 0.20 0.15       NaN  1.357448       NaN
     0.25 0.20       NaN  1.373054       NaN
     0.90 0.05       NaN  1.755392       NaN

[1901 rows x 3 columns]

In [9]:
data_010-data_011

VBM  bandgap       CBM
x    y    z                                
0.00 0.00 0.00  0.409739      0.0  0.409739
          0.05  0.418862      0.0  0.418862
          0.10  0.422876      0.0  0.422876
          0.15  0.423488      0.0  0.423488
          0.20  0.424755      0.0  0.424755
...                  ...      ...       ...
0.95 0.90 0.10       NaN      0.0       NaN
     0.95 0.05       NaN      0.0       NaN
1.00 0.20 0.15       NaN      0.0       NaN
     0.25 0.20       NaN      0.0       NaN
     0.90 0.05       NaN      0.0       NaN

[1901 rows x 3 columns]

In [10]:
min(data_010.CBM-data_011.CBM), max(data_010.CBM-data_011.CBM)

(0.3052341114400283, 0.5798240772933818)

In [14]:
np.argmin(data_010.CBM-data_011.CBM)

np.int64(1394)

In [15]:
data_010.iloc[1394]

VBM       -5.937838
bandgap    1.238870
CBM       -4.698968
Name: (0.9500000000000001, 0.0, 1.0), dtype: float64

In [ ]:
BiSeI

In [16]:
np.argmax(data_010.CBM-data_011.CBM)

np.int64(1755)

In [17]:
data_010.iloc[1755]

VBM       -6.318055
bandgap    1.755376
CBM       -4.562679
Name: (1.0, 1.0, 0.4), dtype: float64

In [ ]:
BiSI_0.4Br_0.6

In [ ]:
data_010.CBM-data_011.CBM

In [36]:
data_010.iloc[1000].name - np.array([0.75, 0, 0.2])

array([0., 0., 0.])

In [43]:
for i in range(len(data_010)):
    if sum(data_010.iloc[i].name - np.array([0.1, 0, 0]))==0:
        print(i, data_010.iloc[i].name, data_010.iloc[i].CBM-data_011.iloc[i].CBM)

2 (np.float64(0.0), np.float64(0.0), np.float64(0.1)) 0.39387635940563026
22 (np.float64(0.0), np.float64(0.05), np.float64(0.05)) 0.3845422552653437
230 (np.float64(0.05), np.float64(0.0), np.float64(0.05)) 0.3798216038623936


In [ ]:
BiSI -> 0.69 eV
BiSeI -> 0.67 eV
BiSBr -> 0.80 eV
BiSeBr -> 0.39 eV
SbSI -> 0.63 eV
SbSeI -> 0.66 eV
SbSBr -> 0.79 eV
SbSeBr -> 0.67 eV